# <a id='toc1_'></a>[Capturing Alternate Abbreviation Symbols Experiment](#toc0_)

Goal of this experiment is to see how well Claude can match my annotations of gene alias pairs in literary context

**Table of contents**<a id='toc0_'></a>    
- [Capturing Alternate Abbreviation Symbols Experiment](#toc1_)    
    - [Import manually curated table](#toc1_1_1_)    
    - [Add the official gene name for each sample from HGNC](#toc1_1_2_)    
    - [Add the abstract and full text for each sample](#toc1_1_3_)    
      - [Not all contexts are programmatically accessible](#toc1_1_3_1_)    
    - [Using details from each sample, generate a custom prompt for each sample](#toc1_1_4_)    
      - [Prompt with just the abstract as context](#toc1_1_4_1_)    
      - [Prompt with full text as context](#toc1_1_4_2_)    
      - [Prompt with full text as context but abstract if full is not available](#toc1_1_4_3_)    
    - [Run prompt through Claude](#toc1_1_5_)    
      - [Run a test with 2 samples (abstracts as context)](#toc1_1_5_1_)    
      - [Run all samples (abstracts as context)](#toc1_1_5_2_)    
      - [Run a test with 2 samples (full text as context)](#toc1_1_5_3_)    
      - [Run a test with 20 samples (full text as context)](#toc1_1_5_4_)    
      - [Run all samples (full text as context)](#toc1_1_5_5_)    
      - [Run a test with 2 samples (available context, prioritizing full text)](#toc1_1_5_6_)    
      - [Run a test with 20 samples (available context, prioritizing full text)](#toc1_1_5_7_)    
    - [Run all samples (available context, prioritizing full text)](#toc1_1_6_)    
    - [Analysis](#toc1_1_7_)    
      - [Analysis (full text as content)](#toc1_1_7_1_)    
      - [Analysis (20 samples, full text as content)](#toc1_1_7_2_)    
      - [Analysis (20 samples, available context, prioritizing full text)](#toc1_1_7_3_)    
      - [Analyze (available context, prioritizing full text)](#toc1_1_7_4_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

In [87]:
import ssl
import certifi
import os

os.environ["SSL_CERT_FILE"] = certifi.where()
ssl._create_default_https_context = ssl.create_default_context

In [88]:
import pandas as pd
import requests
from Bio import Entrez
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()
import boto3
import json
import time
from functools import lru_cache
from urllib.error import HTTPError
from http.client import IncompleteRead
import numpy as np

### <a id='toc1_1_1_'></a>[Import manually curated table](#toc0_)

This is a file where I manually annotated gene alias pairs by answering the following questions:
- Does this alias symbol represent this gene in this publication?
- How does this alias symbol represent this gene in this publication?

**Columns and content description:**
- HGNC_ID, ENSG_ID, NCBI_ID: the identifier(s) associated with the gene in the databases HUGO Gene Nomenclature Committee, Ensembl, and NCBI Gene
- primary_gene_symbol: the official gene symbol assigned to the gene by HGNC
- gene_symbol: the alias symbol that Pubtator3 assigned to the referenced gene in this publication
- captured_status: if the relationship between the alias symbol and the gene is annotated (from gene-harmony-analysis)
- captured_category_list: the list of relationship categories the alias symbol and the gene were annotated by (from gene-harmony-analysis)
- pubtator3_pmid: the PMID of the article that contains the alias symbol 
> **Manually annotated**
- alias_represent_gene_status: if the alias represents the referenced gene in this article
- alternate_abbreviation_status: if the alias represents the referenced gene as an alternate abbreviation in this article
- simple_relationship_type: the relationship between the alias symbol and the referenced gene (possible values include: alias, other, alternate abbreviation, or none)
- detailed_relationship_type: more details about the relationship between the alias symbol and the referenced gene if simple_relationship_type is "other"
- alias_representation_details: what the alias symbol represents if alias_represent_gene_status is false
- relevant_text_section: the section of the article that the relevant text is found in
- relevant_text: the one or two sentences that are evidence for the decisions made in alias_represent_gene_status, alternate_abbreviation_status, simple_relationship_type, detailed_relationship_type, and alias_representation_details.
- notes: miscellaneous thoughts I jotted down as I was curating

In [279]:
#Alternate Abbreviation Symbol Capture- Manually Curated Set
alt_abbrev_annotation_df = pd.read_csv(
    "../input/Alternate Abbreviation Symbol Capture- Manually Curated Set.csv", encoding="latin1", skip_blank_lines=True, true_values=["TRUE", "T"],
    false_values=["FALSE", "F"])
alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,simple_relationship_type,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9477305,True,False,Other,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9390831,True,False,Other,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,7479945,True,False,Other,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,39961042,False,False,NaN,NaN,primer,methods,The primers used in this study were: RD5: 5_ÐA...,NaN
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,38980911,False,False,NaN,NaN,primer,methods,PCR amplification of DNA was performed using a...,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,27999288,True,False,Alias,Alias,NaN,abstract,Melatonin has been speculated to be mainly syn...,NaN
171,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36829878,True,False,Alias,Alias,NaN,abstract,A new clade of serotonin N-acetyltransferase (...,NaN
172,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36009250,True,False,Alias,Alias,NaN,introduction,AANAT is also named serotonin N-acetyltransfe...,NaN
173,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,35424370,False,False,NaN,NaN,NaN,abstract,A polycarbazole-Sn(iv) arsenotungstate (Pcz-Sn...,NaN


### <a id='toc1_1_2_'></a>[Add the official gene name for each sample from HGNC](#toc0_)

In [280]:
def get_gene_name(hgnc_id):
    """ Retreive official gene name from HGNC for each sample in the set.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol of each sample
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    url = f"https://rest.genenames.org/fetch/hgnc_id/{hgnc_id}"
    headers = {"Accept": "application/json"}  # Request JSON format
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        try:
            gene_name = data['response']['docs'][0]['name']
            return gene_name
        except IndexError:
            return f"No gene found for HGNC ID {hgnc_id}"
    else:
        return f"Error: {response.status_code}"

In [281]:
gene_cache = {}

def get_gene_name_cached(hgnc_id):
    """ Retrieve the official gene name from HGNC using a cache to avoid repeated API requests for the same HGNC ID.

    :param hgnc_id: The HGNC ID associated with the primary gene symbol
    :return: The official gene name from HGNC for the passed HGNC ID
    """
    if hgnc_id not in gene_cache:
        gene_cache[hgnc_id] = get_gene_name(hgnc_id)
    return gene_cache[hgnc_id]

In [282]:
alt_abbrev_annotation_df["gene_name"] = alt_abbrev_annotation_df["HGNC_ID"].apply(get_gene_name_cached)
alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,simple_relationship_type,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9477305,True,False,Other,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9390831,True,False,Other,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,7479945,True,False,Other,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,39961042,False,False,NaN,NaN,primer,methods,The primers used in this study were: RD5: 5_ÐA...,NaN,TUB bipartite transcription factor
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,38980911,False,False,NaN,NaN,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,27999288,True,False,Alias,Alias,NaN,abstract,Melatonin has been speculated to be mainly syn...,NaN,aralkylamine N-acetyltransferase
171,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36829878,True,False,Alias,Alias,NaN,abstract,A new clade of serotonin N-acetyltransferase (...,NaN,aralkylamine N-acetyltransferase
172,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36009250,True,False,Alias,Alias,NaN,introduction,AANAT is also named serotonin N-acetyltransfe...,NaN,aralkylamine N-acetyltransferase
173,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,35424370,False,False,NaN,NaN,NaN,abstract,A polycarbazole-Sn(iv) arsenotungstate (Pcz-Sn...,NaN,aralkylamine N-acetyltransferase


### <a id='toc1_1_3_'></a>[Add the abstract and full text for each sample](#toc0_)

In [251]:
Entrez.email = os.environ.get("NCBI_EMAIL")

if not Entrez.email:
    raise RuntimeError(
        'NCBI_EMAIL environment variable not set. '
        'Please set it before running this notebook, \n'
        '  make a .env file and within, assign your email as an environment variable, NCBI_EMAIL="your.email@institution.org"'
    )

# Throttling (≤3 requests/sec without API key)
NCBI_SLEEP = 0.34

# Retry policy
MAX_RETRIES = 5
BASE_BACKOFF = 1.0

In [252]:
def _entrez_call(func):
    """Execute an Entrez call with retry, exponential backoff, and throttling."""
    for attempt in range(MAX_RETRIES):
        try:
            time.sleep(NCBI_SLEEP)
            return func()
        except (RuntimeError, HTTPError, IncompleteRead) as e:
            if attempt == MAX_RETRIES - 1:
                return None
            time.sleep(BASE_BACKOFF * (2 ** attempt))


In [253]:
@lru_cache(maxsize=10000)
def get_abstract(pmid):
    """ Retrieve the abstract text for a PubMed article given its PMID. The abstract is fetched from the NCBI PubMed database using Entrez.
    
    If the abstract consists of multiple sections, they are concatenated
    into a single string separated by blank lines.

    :param pmid: The PubMed ID (PMID) of the article
    :return: The abstract text as a string, or None if no abstract is available
    """
    pmid = str(pmid)

    def call():
        handle = Entrez.efetch(
            db="pubmed", id=pmid, rettype="abstract", retmode="xml"
        )
        records = Entrez.read(handle)
        handle.close()
        return records

    records = _entrez_call(call)
    if not records:
        return None

    try:
        abstract = (
            records["PubmedArticle"][0]
            ["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
        )
        return "\n\n".join(abstract) if isinstance(abstract, list) else abstract
    except (KeyError, IndexError, TypeError):
        return None


In [254]:
def get_abstract_with_abbreviations(pmid):
    """
    Retrieve abstract text and prepend the abbreviations section if available
    from PMC XML.
    """
    pmid = str(pmid)

    # Step 1: Try to get abbreviations from PMC XML
    abbreviations = []
    try:
        # Fetch PMC XML
        def elink_call():
            handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
            records = Entrez.read(handle)
            handle.close()
            return records

        records = _entrez_call(elink_call)
        if records:
            try:
                pmcid = records[0]["LinkSetDb"][0]["Link"][0]["Id"]
                
                def efetch_call():
                    handle = Entrez.efetch(
                        db="pmc", id=pmcid, rettype="full", retmode="xml"
                    )
                    xml = handle.read()
                    handle.close()
                    return xml
                
                xml = _entrez_call(efetch_call)
                if xml and b"does not allow downloading" not in xml:
                    soup = BeautifulSoup(xml, "xml")
                    def_list = soup.find("def-list", {"list-content": "abbreviations"})
                    if def_list:
                        for item in def_list.find_all("def-item"):
                            term = item.find("term")
                            definition = item.find("def")
                            if term and definition:
                                term_text = term.get_text(strip=True)
                                def_text = definition.get_text(" ", strip=True)
                                abbreviations.append(f"{term_text} = {def_text}")
            except Exception:
                pass
    except Exception:
        pass

    # Step 2: Fetch abstract
    abstract_text = get_abstract(pmid)
    if abstract_text is None:
        return None

    # Step 3: Prepend abbreviations if they exist
    if abbreviations:
        abbrev_paragraph = "ABBREVIATIONS:\n" + "\n".join(abbreviations)
        abstract_text = abbrev_paragraph + "\n\n" + abstract_text

    return abstract_text


In [255]:
@lru_cache(maxsize=10000)
def get_full_text_paragraphs(pmid):
    """ Retrieve full-text paragraphs for a PubMed article when available via PMC.

    This function attempts to map a PubMed ID (PMID) to a PubMed Central ID
    (PMCID). If a PMCID exists and the article allows XML access, the full text
    is fetched from PMC in XML format and parsed into individual paragraph
    strings.

    :param pmid: pmid : The PubMed ID (PMID) of the article.
    :return: A list of paragraph texts extracted from the PMC full text, or None
    if the article is not available in PMC, access is restricted, or an
    error occurs during retrieval.
    """
    pmid = str(pmid)

    # Step 1: PMID → PMCID
    def elink_call():
        handle = Entrez.elink(dbfrom="pubmed", db="pmc", id=pmid)
        records = Entrez.read(handle)
        handle.close()
        return records

    records = _entrez_call(elink_call)
    if not records:
        return None

    try:
        pmcid = records[0]["LinkSetDb"][0]["Link"][0]["Id"]
    except (IndexError, KeyError, TypeError):
        return None

    # Step 2: Fetch PMC XML
    def efetch_call():
        handle = Entrez.efetch(
            db="pmc", id=pmcid, rettype="full", retmode="xml"
        )
        xml = handle.read()
        handle.close()
        return xml

    xml = _entrez_call(efetch_call)
    if not xml:
        return None

    if b"does not allow downloading of the full text in XML form" in xml:
        return None

    try:
        soup = BeautifulSoup(xml, "xml")
        # extract abbreviations first
        abbreviations = []
        def_list = soup.find("def-list", {"list-content": "abbreviations"})
        if def_list:
            for item in def_list.find_all("def-item"):
                term = item.find("term")
                definition = item.find("def")
                if term and definition:
                    term_text = term.get_text(strip=True)
                    def_text = definition.get_text(" ", strip=True)
                    abbreviations.append(f"{term_text} = {def_text}")

        # Turn abbreviations into a single paragraph
        abbrev_paragraph = None
        if abbreviations:
            abbrev_paragraph = "ABBREVIATIONS:\n" + "\n".join(abbreviations)

        # extract regular paragraphs
        paragraphs = [p.get_text(strip=True) for p in soup.find_all("p")]

        # combine abbreviations at the top
        if abbrev_paragraph:
            paragraphs = [abbrev_paragraph] + paragraphs

        return paragraphs or None

    except Exception:
        return None


In [256]:
def chunk_paragraphs(paragraphs, max_chars=20000):
    """ Chunk paragraph text into size-limited chunks.
    Paragraph boundaries are preserved; paragraphs are not split
    across chunks.

    :param paragraphs: A list of paragraph strings to be grouped into chunks
    :param max_chars: Maximum number of characters allowed per chunk
    :return: A list of text chunks, each containing one or more paragraphs
    """
    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) <= max_chars:
            current += p + "\n\n"
        else:
            chunks.append(current.strip())
            current = p + "\n\n"

    if current:
        chunks.append(current.strip())

    return chunks


In [257]:
def get_full_text_chunks(pmid, max_chars=20000):
    """ Retrieve chunked full-text content of a PubMed article via PMC.

    This function retrieves the full-text paragraphs for a given PubMed ID
    (PMID) when available via PubMed Central (PMC), and groups those paragraphs
    into size-limited text chunks suitable for downstream processing such as
    LLM prompting.

    :param pmid: The PubMed ID (PMID) of the article
    :param max_chars: Maximum number of characters allowed per text chunk
    :return: A list of text chunks derived from the article full text, or None
             if the article is unavailable, restricted, or an error occurs
    """
    try:
        paragraphs = get_full_text_paragraphs(pmid)
        if not paragraphs:
            return None
        return chunk_paragraphs(paragraphs, max_chars=max_chars)
    except Exception:
        return None


In [100]:
# #Output already generated! Only need to rerun if there are changes made to alt_abbrev_annotation_df
# alt_abbrev_annotation_using_fulltext_df = alt_abbrev_annotation_df.copy()


# alt_abbrev_annotation_using_fulltext_df["pmid_full_text_chunks"] = (
#     alt_abbrev_annotation_using_fulltext_df["pubtator3_pmid"]
#     .progress_apply(get_full_text_chunks)
# )

100%|██████████| 175/175 [06:42<00:00,  2.30s/it]


In [101]:
# #Output already generated! Only need to rerun if there are changes made to alt_abbrev_annotation_df
# alt_abbrev_annotation_using_fulltext_df = (
#     alt_abbrev_annotation_using_fulltext_df
#     .explode("pmid_full_text_chunks")
#     .rename(columns={"pmid_full_text_chunks": "pmid_full_text_chunk"})
# )
# alt_abbrev_annotation_using_fulltext_df    

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,simple_relationship_type,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,TRUE,FALSE,Other,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,None
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,TRUE,FALSE,Other,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,TRUE,FALSE,Other,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact..."
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,TRUE,FALSE,Other,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,None
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,39961042,FALSE,FALSE,NaN,NaN,primer,methods,The primers used in this study were: RD5: 5_ÐA...,NaN,TUB bipartite transcription factor,Blastocystissp. is a zoonotic intestinal proto...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
172,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,F,NaN,36009250,TRUE,FALSE,Alias,Alias,NaN,introduction,AANAT is also named serotonin N-acetyltransfe...,NaN,aralkylamine N-acetyltransferase,Melatonin has diverse functions in plant growt...
173,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,F,NaN,35424370,FALSE,FALSE,NaN,NaN,NaN,abstract,A polycarbazole-Sn(iv) arsenotungstate (Pcz-Sn...,NaN,aralkylamine N-acetyltransferase,Electronic supplementary information (ESI) ava...
173,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,F,NaN,35424370,FALSE,FALSE,NaN,NaN,NaN,abstract,A polycarbazole-Sn(iv) arsenotungstate (Pcz-Sn...,NaN,aralkylamine N-acetyltransferase,It can be seen in the TGA graph that at 650 °C...
174,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,F,NaN,35326246,TRUE,FALSE,Alias,Alias,NaN,abstract,Serotonin N-acetyltransferase is the penultima...,NaN,aralkylamine N-acetyltransferase,SerotoninN-acetyltransferase is the penultimat...


In [102]:
# #Output already generated! Only need to rerun if there are changes made to alt_abbrev_annotation_df
# alt_abbrev_annotation_using_fulltext_df.to_excel('../output/alt_abbrev_annotation_using_fulltext_df.xlsx',index=False)

In [103]:
alt_abbrev_annotation_using_fulltext_df = pd.read_excel("../output/alt_abbrev_annotation_using_fulltext_df.xlsx")

In [107]:
# #Output already generated! Only need to rerun if there are changes made to alt_abbrev_annotation_df
# alt_abbrev_annotation_using_abstract_df = alt_abbrev_annotation_df.copy()

# alt_abbrev_annotation_using_abstract_df["abstract_pmid_text"] = (
#     alt_abbrev_annotation_using_abstract_df["pubtator3_pmid"]
#     .progress_apply(get_abstract)
# )

100%|██████████| 175/175 [01:40<00:00,  1.75it/s]


In [108]:
# #Output already generated! Only need to rerun if there are changes made to alt_abbrev_annotation_df
# alt_abbrev_annotation_using_abstract_df.to_excel('../output/alt_abbrev_annotation_using_abstract_df.xlsx',index=False)

In [109]:
alt_abbrev_annotation_using_abstract_df = pd.read_excel("../output/alt_abbrev_annotation_using_abstract_df.xlsx")

#### <a id='toc1_1_3_1_'></a>[Not all contexts are programmatically accessible](#toc0_)

In [110]:
num_missing_abstracts = len(
    alt_abbrev_annotation_using_abstract_df.loc[
        alt_abbrev_annotation_using_abstract_df["abstract_pmid_text"].isna()
    ]
)

num_missing_full_texts = len(
    alt_abbrev_annotation_using_fulltext_df.loc[
        alt_abbrev_annotation_using_fulltext_df["pmid_full_text_chunk"].isna()
    ]
)

print(f"Number of abstracts that are not accessible: {num_missing_abstracts}")
print(f"Number of full texts that are not accessible: {num_missing_full_texts}")

Number of abstracts that are not accessible: 3
Number of full texts that are not accessible: 21


In [123]:
df = alt_abbrev_annotation_using_fulltext_df.loc[
        alt_abbrev_annotation_using_fulltext_df["pmid_full_text_chunk"].isna()
    ]
df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,simple_relationship_type,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,True,False,Other,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,True,False,Other,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN
15,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,9116635,False,False,NaN,NaN,virus glycoprotein 58,abstract,HCMV DNA encoding a neutralising epitope of th...,NaN,"lectin, mannose binding 1",NaN
20,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,8383182,False,False,NaN,NaN,gp58/116 complex of human cytomegalovirus (HCMV),abstract,The gene for the homologue of herpesvirus glyc...,NaN,"lectin, mannose binding 1",NaN
24,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,7688304,False,False,NaN,NaN,human cytomegalovirus 58-kDa glycoprotein,abstract,"We have mapped continuous epitopes, for positi...",NaN,"lectin, mannose binding 1",NaN
25,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,7678304,False,False,NaN,NaN,human cytomegalovirus (CMV) membrane glycoprot...,abstract,The humoral immune response to human cytomegal...,NaN,"lectin, mannose binding 1",NaN
26,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,7521934,False,False,NaN,NaN,human cytomegalovirus glycoprotein B (gp58/116),abstract,The nucleotide sequences of the variable regio...,NaN,"lectin, mannose binding 1",NaN
40,HGNC:9911,ENSG00000216835,GENE ID:3186,RBMXP1,HNRNP-G,F,NaN,7692398,True,False,Other,derivative of HGNC Previous Symbol,NaN,abstract,Here we show that p43 is an RNA-binding protei...,NaN,RBMX pseudogene 1,NaN
109,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,F,NaN,9799223,True,False,Other,literature defined (uniprot alternative protei...,NaN,abstract,We now report the purification of a 150 kDa re...,NaN,BAG cochaperone 6,NaN
167,HGNC:74,ENSG00000118777,GENE ID:9429,ABCG2,CD338,T,Prefix Gene Group Symbol,38513324,True,False,Other,HCDM antigen nomenclature,NaN,materials and methods,The 5D3 primary antibody (Purified mouse anti-...,NaN,ATP binding cassette subfamily G member 2 (JR ...,NaN


### <a id='toc1_1_4_'></a>[Using details from each sample, generate a custom prompt for each sample](#toc0_)

In [111]:
def generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name):
    """ Generate an LLM prompt for curating alias gene symbols and determining if their relationship to an official HGNC gene record is an alternate abbreviation.

    The generated prompt instructs the model to:
    - Classify the relationship as an Alternate Abbreviation or not
    - Extract exact textual evidence from a biomedical context
    - Return the result as a strictly formatted JSON object

    :param alias_symbol: The alias gene symbol to be evaluated
    :param hgnc_id: The HGNC record ID associated with the primary gene
    :param primary_gene_symbol: The official HGNC gene symbol
    :param name: The official HGNC gene name
    :return: A formatted prompt string for use with a language model
    """
    prompt = f"""Role:You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias resolution.

    Inputs:
    Alias gene symbol: {alias_symbol}
    HGNC record ID: {hgnc_id}
    Primary gene symbol: {primary_gene_symbol}
    Official HGNC gene name: {name}
    Biomedical text context

    Task:
    Determine whether the alias symbol in the given text represents the gene associated with the provided HGNC record ID
    as an alternate abbreviation of the official HGNC gene symbol.

    Classify the alias symbol as an Alternate Abbreviation only if it is expanded in the text to the official
    HGNC gene name, or to a phrase that is nearly identical in wording and meaning.

    The mere presence of both the alias symbol and the official HGNC gene name in the text is not sufficient. 
    The text must link the alias symbol to the official gene name through an expansion.

    Examples:
    1-
    Text: “A 64-kDa human protein, cloned and identified as either ACF (APOBEC-1complementation factor) (8) or 
    ASP (APOBEC-1 stimulating protein) (9), together with APOBEC-1 function as a minimal editosome in vitro.”
    Reasoning: ACF64 and ACF65 are used like this "ACF64/ACF65" in the text and therefore it can be assumed that
    this sentence is also applicable to ACF65.
    Alias gene symbol: ACF65
    Official HGNC gene name: APOBEC1 complementation factor
    Primary gene symbol: A1CF
    2-
    Text: “Editing of apolipoprotein (apo) B mRNA is mediated by an enzyme-complex that consists of the catalytic 
    cytidine deaminase APOBEC-1 and the mRNA binding protein APOBEC-1 complementation factor or APOBEC-1 stimulating protein (ACF/ASP).”
    Reasoning: ASP is represented by APOBEC-1 stimulating protein where stimulating is a synonym for complementation
    Alias gene symbol: ASP
    Official HGNC gene name: APOBEC1 complementation factor
    Primary gene symbol: A1CF
    3-
    Text: In the abbreviations section “α4GnT α1,4-N-acetylglucosaminyltransferase”
    Reasoning: ALPHA4GNT is the same as A4GNT but with the term alpha spelled out instead of just shortened to an A
    Alias gene symbol: ALPHA4GNT
    Official HGNC gene name: alpha-1,4-N-acetylglucosaminyltransferase
    Primary gene symbol: A4GNT
    
    Rules:
    If the text does not support that the alias is an alternate abbreviation, return:
    "alternate_abbreviation": "FALSE"
    "evidence": ""

    If the text does support that the alias is an alternate abbreviation, set "alternate_abbreviation": "TRUE"

    Evidence must be one exact sentence copied verbatim from the text.
    Do not infer relationships or paraphrase evidence.

    Output (Strict):
    Output only one JSON object. No markdown.
    {{
    "pmid": "",
    "alias_symbol": "{alias_symbol}",
    "primary_gene_symbol": "{primary_gene_symbol}",
    "name": "{name}",
    "alternate_abbreviation": "TRUE or FALSE",
    "evidence": ""
    }}"""

    return prompt

In [267]:
def generate_alt_abbrev_prompt_w_no_context(alias_symbol, hgnc_id, primary_gene_symbol, name):
    """ Generate an LLM prompt for curating alias gene symbols and determining if their relationship to an official HGNC gene record is an alternate abbreviation.

    The generated prompt instructs the model to:
    - Classify the relationship as an Alternate Abbreviation or not

    :param alias_symbol: The alias gene symbol to be evaluated
    :param hgnc_id: The HGNC record ID associated with the primary gene
    :param primary_gene_symbol: The official HGNC gene symbol
    :param name: The official HGNC gene name
    :return: A formatted prompt string for use with a language model
    """
    prompt = f"""Role:You are a biomedical gene nomenclature curator trained in HGNC gene identity and alias symbol resolution.

    Inputs:
    Alias gene symbol: {alias_symbol}
    HGNC record ID: {hgnc_id}
    Primary gene symbol: {primary_gene_symbol}
    Official HGNC gene name: {name}

    Task:
    Determine whether the alias symbol is an abbreviation variant of the official HGNC gene name. 
    Classify the alias symbol as an Alternate Abbreviation only if the letters are reasonably able to be expanded to the official
    HGNC gene name, or to a phrase that is nearly identical in wording and meaning.

    Examples:
    1-
    Alias gene symbol: ACF65
    Official HGNC gene name: APOBEC1 complementation factor
    Primary gene symbol: A1CF
    2-
    Alias gene symbol: ALPHA4GNT
    Official HGNC gene name: alpha-1,4-N-acetylglucosaminyltransferase
    Primary gene symbol: A4GNT
    
    Rules:
    If the text does not support that the alias is an alternate abbreviation, return:
    "alternate_abbreviation": "FALSE"

    If the text does support that the alias is an alternate abbreviation, set "alternate_abbreviation": "TRUE"

    Output (Strict):
    Output only one JSON object. No markdown.
    {{
    "pmid": "",
    "alias_symbol": "{alias_symbol}",
    "primary_gene_symbol": "{primary_gene_symbol}",
    "name": "{name}",
    "alternate_abbreviation": "TRUE or FALSE"
    }}"""

    return prompt

#### <a id='toc1_1_4_1_'></a>[Prompt with just the abstract as context](#toc0_)

In [112]:
# alt_abbrev_annotation_with_prompt_using_abstract_df = alt_abbrev_annotation_using_abstract_df.copy()
# alt_abbrev_annotation_with_prompt_using_abstract_df['context'] = None
# alt_abbrev_annotation_with_prompt_using_abstract_df['alt_abbrev_prompt'] = None

In [113]:
# for idx, row in alt_abbrev_annotation_with_prompt_using_abstract_df.iterrows():
#     alias_symbol = row['gene_symbol']
#     primary_gene_symbol = row['primary_gene_symbol']
#     name = row['gene_name']
#     hgnc_id = row['HGNC_ID']
#     prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

#     context = f'PMID: {row['pubtator3_pmid']}\n Abstract: {row['abstract_pmid_text']}'

#     full_prompt = f'{prompt_base}\n\n{context}'

#     alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'context'] = context
#     alt_abbrev_annotation_with_prompt_using_abstract_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



# alt_abbrev_annotation_with_prompt_using_abstract_df.head()

#### <a id='toc1_1_4_2_'></a>[Prompt with full text as context](#toc0_)

In [114]:
# alt_abbrev_annotation_with_prompt_using_fulltext_df = alt_abbrev_annotation_using_fulltext_df.copy()
# alt_abbrev_annotation_with_prompt_using_fulltext_df['context'] = None
# alt_abbrev_annotation_with_prompt_using_fulltext_df['alt_abbrev_prompt'] = None

In [115]:
# alt_abbrev_annotation_with_prompt_using_fulltext_df = (
#     alt_abbrev_annotation_with_prompt_using_fulltext_df.reset_index(drop=True)
# )

# for idx, row in alt_abbrev_annotation_with_prompt_using_fulltext_df.iterrows():
#     alias_symbol = row['gene_symbol']
#     primary_gene_symbol = row['primary_gene_symbol']
#     name = row['gene_name']
#     hgnc_id = row['HGNC_ID']
#     prompt_base = generate_alt_abbrev_prompt(alias_symbol, hgnc_id, primary_gene_symbol, name)

#     context = f'PMID: {row['pubtator3_pmid']}\n Full Text: {row['pmid_full_text_chunk']}'

#     full_prompt = f'{prompt_base}\n\n{context}'

#     alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'context'] = context
#     alt_abbrev_annotation_with_prompt_using_fulltext_df.at[idx,'alt_abbrev_prompt'] = full_prompt    



# alt_abbrev_annotation_with_prompt_using_fulltext_df.head()

#### <a id='toc1_1_4_3_'></a>[Prompt with full text as context but abstract if full is not available](#toc0_)

In [116]:
alt_abbrev_annotation_using_fulltext_and_abstract_df = alt_abbrev_annotation_using_fulltext_df.merge(alt_abbrev_annotation_using_abstract_df, on=["HGNC_ID","ENSG_ID","NCBI_ID","primary_gene_symbol","gene_symbol","captured_status","captured_category_list","pubtator3_pmid","alias_represent_gene_status","alternate_abbreviation_status","simple_relationship_type","detailed_relationship_type","alias_representation_details","relevant_text_section","relevant_text","notes","gene_name"], how="outer")

In [117]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = alt_abbrev_annotation_using_fulltext_and_abstract_df.copy()
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['context'] = None
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_prompt'] = None

In [118]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = (
    alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.reset_index(drop=True)
)

for idx, row in alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.iterrows():
    alias_symbol = row["gene_symbol"]
    primary_gene_symbol = row["primary_gene_symbol"]
    name = row["gene_name"]
    hgnc_id = row["HGNC_ID"]

    prompt_base = generate_alt_abbrev_prompt(
        alias_symbol, hgnc_id, primary_gene_symbol, name
    )

    full_text = row["pmid_full_text_chunk"]

    if isinstance(full_text, str) and full_text.strip():
        source = "Full Text"
        text = full_text
    else:
        source = "Abstract"
        text = row["abstract_pmid_text"]

    context = f'PMID: {row["pubtator3_pmid"]}\n{source}: {text}'
    full_prompt = f"{prompt_base}\n\n{context}"

    alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.at[idx, "context"] = context
    alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.at[idx, "alt_abbrev_prompt"] = full_prompt

alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,True,False,...,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN,Usher syndrome is a group of diseases with aut...,PMID: 7479945\nAbstract: Usher syndrome is a g...,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,True,False,...,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\nFull Text: These authors contri...,Role:You are a biomedical gene nomenclature cu...
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,True,False,...,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...",Mice homozygous for a defect of the tub (rd5) ...,"PMID: 9390831\nFull Text: In this paper, we co...",Role:You are a biomedical gene nomenclature cu...
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,True,False,...,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN,Some genetic syndromes causing loss of hearing...,PMID: 9477305\nAbstract: Some genetic syndrome...,Role:You are a biomedical gene nomenclature cu...
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,36612533,False,False,...,NaN,primer,methods,The primers were as follows: forward primer RD...,NaN,TUB bipartite transcription factor,Blastocystisis one of the most common enteric ...,<i>Blastocystis</i> is one of the most common ...,PMID: 36612533\nFull Text: Blastocystisis one ...,Role:You are a biomedical gene nomenclature cu...


There are 7 context chunks with weird formatting notation. Since only 7 and its mixed with other useful text, I will keep these chunks and do nothing about it for now.

In [119]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['context'].str.contains('usepackage', case=False, na=False).sum()

7

In [120]:
usepackage_filtered_df = alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df[alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['context'].str.contains('usepackage', na=False)]
usepackage_filtered_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
77,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,GB3S,F,NaN,32948883,True,False,...,Alias,plural of gb3,abstract,Gb3 glycosphingolipids are the specific recept...,NaN,"alpha 1,4-galactosyltransferase (P1PK blood gr...","STxB binding to the different Gb3species, exce...",Gb<sub>3</sub> glycosphingolipids are the spec...,PMID: 32948883\nFull Text: STxB binding to the...,Role:You are a biomedical gene nomenclature cu...
276,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,30635007,False,False,...,NaN,initials of Guanai Bao,authorsÕ contribution,"JZ, RR, GAB, XMS, YLJ, JYD, JFF, and NN partic...",NaN,alpha-1-B glycoprotein,Transcutaneous electrical acupoint stimulation...,Transcutaneous electrical acupoint stimulation...,PMID: 30635007\nFull Text: Transcutaneous elec...,Role:You are a biomedical gene nomenclature cu...
295,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,40567688,False,False,...,NaN,global attention branch (GAB),methodology,"Therefore, a dual-branch global extraction mod...",NaN,alpha-1-B glycoprotein,"Transformer (Vaswani et al., 2017) was initial...",The semantic segmentation task of remote sensi...,PMID: 40567688\nFull Text: Transformer (Vaswan...,Role:You are a biomedical gene nomenclature cu...
296,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,40567688,False,False,...,NaN,global attention branch (GAB),methodology,"Therefore, a dual-branch global extraction mod...",NaN,alpha-1-B glycoprotein,"Next, the generated self-attention weight matr...",The semantic segmentation task of remote sensi...,"PMID: 40567688\nFull Text: Next, the generated...",Role:You are a biomedical gene nomenclature cu...
297,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,40567688,False,False,...,NaN,global attention branch (GAB),methodology,"Therefore, a dual-branch global extraction mod...",NaN,alpha-1-B glycoprotein,(2) Effect of CIIM module: The addition of the...,The semantic segmentation task of remote sensi...,PMID: 40567688\nFull Text: (2) Effect of CIIM ...,Role:You are a biomedical gene nomenclature cu...
320,HGNC:6631,ENSG00000074695,GENE ID:3998,LMAN1,GP58,F,NaN,7506283,False,False,...,NaN,glycoproteins gp58/116 (gB),abstract,"As antigens, three immunodominant domains capa...",NaN,"lectin, mannose binding 1",HIV-1 infection has been associated with incre...,The IgG subclass pattern against linear antibo...,PMID: 7506283\nFull Text: HIV-1 infection has ...,Role:You are a biomedical gene nomenclature cu...
409,HGNC:74,ENSG00000118777,GENE ID:9429,ABCG2,CD338,T,Prefix Gene Group Symbol,40890122,True,False,...,HCDM antigen nomenclature,NaN,introduction,"CSCs, compared to differentiated tumors found ...",NaN,ATP binding cassette subfamily G member 2 (JR ...,"Overall, these results suggested that sNK cell...",This study highlights the significance of supe...,"PMID: 40890122\nFull Text: Overall, these resu...",Role:You are a biomedical gene nomenclature cu...


In [121]:
df = alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.loc[alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["pubtator3_pmid"] == 35959971]
df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
55,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,F,NaN,35959971,True,True,...,Alternate Abbreviation,NaN,abstract,"alpha4GnT alpha1,4-N-acetylglucosaminyltransfe...",NaN,"alpha-1,4-N-acetylglucosaminyltransferase","ABBREVIATIONS:\nαGlcNAc = α1,4‐linked N ‐acety...",Gastric cancer is the second leading cause of ...,PMID: 35959971\nFull Text: ABBREVIATIONS:\nαGl...,Role:You are a biomedical gene nomenclature cu...
56,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,F,NaN,35959971,True,True,...,Alternate Abbreviation,NaN,abstract,"alpha4GnT alpha1,4-N-acetylglucosaminyltransfe...",NaN,"alpha-1,4-N-acetylglucosaminyltransferase","Given these findings, we focused primarily on ...",Gastric cancer is the second leading cause of ...,PMID: 35959971\nFull Text: Given these finding...,Role:You are a biomedical gene nomenclature cu...


In [134]:
df = alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.loc[alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["pubtator3_pmid"] == 7522117]
df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
298,HGNC:6445,ENSG00000135480,GENE ID:3855,KRT7,SCL,T,Withdrawn MGI Symbol,7522117,True,False,...,literature defined (HGNC alias names),NaN,abstract,"The continued replication of these cells, alth...",NaN,keratin 7,NaN,Prolonged interferon (IFN) treatment can conve...,PMID: 7522117\nAbstract: Prolonged interferon ...,Role:You are a biomedical gene nomenclature cu...


In [136]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.loc[
    alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["context"].str.contains("ABBREVIATIONS", na=False)
]

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
16,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,F,NaN,37879364,True,False,...,literature defined (uniprot alternative protei...,NaN,discussion,PBF also interacts with 14_3_3_ and Scythe/BAT...,NaN,BAG cochaperone 6,ABBREVIATIONS:\nCTL = cytotoxic T lymphocyte\n...,We previously identified papillomavirus bindin...,PMID: 37879364\nFull Text: ABBREVIATIONS:\nCTL...,Role:You are a biomedical gene nomenclature cu...
55,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,F,NaN,35959971,True,True,...,Alternate Abbreviation,NaN,abstract,"alpha4GnT alpha1,4-N-acetylglucosaminyltransfe...",NaN,"alpha-1,4-N-acetylglucosaminyltransferase","ABBREVIATIONS:\nαGlcNAc = α1,4‐linked N ‐acety...",Gastric cancer is the second leading cause of ...,PMID: 35959971\nFull Text: ABBREVIATIONS:\nαGl...,Role:You are a biomedical gene nomenclature cu...
249,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,31837028,False,False,...,NaN,red blood cell type,methods,We used in vitro phagocytosis by monocytes and...,NaN,alpha-1-B glycoprotein,ABBREVIATIONS:\nFDA = US Food and Drug Adminis...,Immunoglobulin therapy including intravenous i...,PMID: 31837028\nFull Text: ABBREVIATIONS:\nFDA...,Role:You are a biomedical gene nomenclature cu...
301,HGNC:6445,ENSG00000135480,GENE ID:3855,KRT7,SCL,T,Withdrawn MGI Symbol,37667768,False,False,...,NaN,spindle cell lipoma,introduction,SCL is a rare benign tumor arising from mature...,NaN,keratin 7,ABBREVIATIONS:\nACD‐RCC = acquired cystic dise...,Collision tumors are a rare phenomenon defined...,PMID: 37667768\nFull Text: ABBREVIATIONS:\nACD...,Role:You are a biomedical gene nomenclature cu...
357,HGNC:7067,ENSG00000179583,GENE ID:4261,CIITA,NLRA,T,Prefix Gene Group Symbol,40613780,True,False,...,Alias,NaN,introduction,NLRs can be further divided into five subfamil...,NaN,class II major histocompatibility complex tran...,ABBREVIATIONS:\nABHD = α/β hydrolase domain\nA...,"Over the past decade, S-acylation has emerged ...",PMID: 40613780\nFull Text: ABBREVIATIONS:\nABH...,Role:You are a biomedical gene nomenclature cu...


#### Prompt that doesn't require context

In [283]:
alt_abbrev_annotation_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,simple_relationship_type,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9477305,True,False,Other,mouse phenotype biomarker,NaN,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,9390831,True,False,Other,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,7479945,True,False,Other,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,39961042,False,False,NaN,NaN,primer,methods,The primers used in this study were: RD5: 5_ÐA...,NaN,TUB bipartite transcription factor
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,False,NaN,38980911,False,False,NaN,NaN,primer,methods,PCR amplification of DNA was performed using a...,NaN,TUB bipartite transcription factor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
170,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,27999288,True,False,Alias,Alias,NaN,abstract,Melatonin has been speculated to be mainly syn...,NaN,aralkylamine N-acetyltransferase
171,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36829878,True,False,Alias,Alias,NaN,abstract,A new clade of serotonin N-acetyltransferase (...,NaN,aralkylamine N-acetyltransferase
172,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,36009250,True,False,Alias,Alias,NaN,introduction,AANAT is also named serotonin N-acetyltransfe...,NaN,aralkylamine N-acetyltransferase
173,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,False,NaN,35424370,False,False,NaN,NaN,NaN,abstract,A polycarbazole-Sn(iv) arsenotungstate (Pcz-Sn...,NaN,aralkylamine N-acetyltransferase


In [284]:
alt_abbrev_annotation_with_prompt_no_context_df = alt_abbrev_annotation_df.copy()
consolidated_alt_abbrev_annotation_with_prompt_no_context_df = (
    alt_abbrev_annotation_with_prompt_no_context_df
    .groupby(["HGNC_ID", "ENSG_ID", "NCBI_ID", "primary_gene_symbol", "gene_symbol", "gene_name"], as_index=False).agg({
        "alternate_abbreviation_status": "any"
    })
)
consolidated_alt_abbrev_annotation_with_prompt_no_context_df['alt_abbrev_prompt'] = None

In [286]:
for idx, row in consolidated_alt_abbrev_annotation_with_prompt_no_context_df.iterrows():
    alias_symbol = row['gene_symbol']
    primary_gene_symbol = row['primary_gene_symbol']
    name = row['gene_name']
    hgnc_id = row['HGNC_ID']
    prompt = generate_alt_abbrev_prompt_w_no_context(alias_symbol, hgnc_id, primary_gene_symbol, name)

    consolidated_alt_abbrev_annotation_with_prompt_no_context_df.at[idx,'alt_abbrev_prompt'] = prompt    



consolidated_alt_abbrev_annotation_with_prompt_no_context_df.head()

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,gene_name,alternate_abbreviation_status,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,TUB bipartite transcription factor,False,Role:You are a biomedical gene nomenclature cu...
1,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,BAG cochaperone 6,False,Role:You are a biomedical gene nomenclature cu...
2,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,"alpha-1,4-N-acetylglucosaminyltransferase",True,Role:You are a biomedical gene nomenclature cu...
3,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,A4GALT1,"alpha 1,4-galactosyltransferase (P1PK blood gr...",True,Role:You are a biomedical gene nomenclature cu...
4,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,GB3S,"alpha 1,4-galactosyltransferase (P1PK blood gr...",False,Role:You are a biomedical gene nomenclature cu...


### <a id='toc1_1_5_'></a>[Run prompt through Claude](#toc0_)

login via aws sso login in terminal

In [137]:
# Initialize the Bedrock Runtime client
bedrock = boto3.client("bedrock-runtime", region_name="us-east-2")

# Replace with your actual inference profile ID or ARN
INFERENCE_PROFILE_ID = "us.anthropic.claude-3-5-sonnet-20240620-v1:0"


def query_claude_sonnet(prompt: str) -> str:
    """ Send a prompt to the Claude Sonnet model via Amazon Bedrock and return the generated response text.

    :param prompt: The input prompt to send to the Claude Sonnet model
    :return: The model-generated response text, or an error message string
    """

    body = {
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "max_tokens": 1024,
        "temperature": 0.0,
        "anthropic_version": "bedrock-2023-05-31"
    }

    try:
        response = bedrock.invoke_model(
            body=json.dumps(body),
            modelId=INFERENCE_PROFILE_ID,
            contentType="application/json",
            accept="application/json"
        )
        response_body = json.loads(response["body"].read())
        return response_body["content"][0]["text"]
    except Exception as e:
        return f"[Error] {str(e)}"

In [183]:
def query_using_dataframe(df):
    # Work on a copy so the original isn't modified
    df_copy = df.copy()
    df_copy['alt_abbrev_response'] = None

    for idx, row in df_copy.iterrows():
        # Skip rows where there is no context available
        if pd.isna(row['context']):
            continue

        df_copy.at[idx, 'alt_abbrev_response'] = query_claude_sonnet(
            row['alt_abbrev_prompt']
        )

        print(f'{idx} Done')

    return df_copy

#### <a id='toc1_1_5_6_'></a>[Run a test with 2 samples (available context, prioritizing full text)](#toc0_)

In [44]:
test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.iloc[:2]
test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,True,False,...,mouse phenotype biomarker,NaN,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN,Usher syndrome is a group of diseases with aut...,PMID: 7479945\nAbstract: Usher syndrome is a g...,Role:You are a biomedical gene nomenclature cu...
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,True,False,...,mouse phenotype biomarker,NaN,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\nFull Text: These authors contri...,Role:You are a biomedical gene nomenclature cu...


In [45]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'] = None

# for idx, row in test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.iterrows():
#     # Skip rows where there is no context available
#     if pd.isna(row['context']):
#         continue

#     test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_52587/1518931753.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'] = None


0 Done
1 Done


In [46]:
# #Comment out after run!
# test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.to_excel('../output/test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx',index=False)

In [47]:
test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = pd.read_excel("../output/test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx")

In [48]:
print(test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'][0])

{
"pmid": "7479945",
"alias_symbol": "rd5",
"primary_gene_symbol": "TUB",
"name": "TUB bipartite transcription factor",
"alternate_abbreviation": "FALSE",
"evidence": ""
}


In [49]:
print(test_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'][1])

{
"pmid": "9390831",
"alias_symbol": "rd5",
"primary_gene_symbol": "TUB",
"name": "TUB bipartite transcription factor",
"alternate_abbreviation": "FALSE",
"evidence": ""
}


#### <a id='toc1_1_5_7_'></a>[Run a test with 20 samples (available context, prioritizing full text)](#toc0_)

In [159]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = (
    alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
        .sample(n=20, random_state=42)
)
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,detailed_relationship_type,alias_representation_details,relevant_text_section,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt
55,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,F,NaN,35959971,True,True,...,Alternate Abbreviation,NaN,abstract,"alpha4GnT alpha1,4-N-acetylglucosaminyltransfe...",NaN,"alpha-1,4-N-acetylglucosaminyltransferase","ABBREVIATIONS:\nαGlcNAc = α1,4‐linked N ‐acety...",Gastric cancer is the second leading cause of ...,PMID: 35959971\nFull Text: ABBREVIATIONS:\nαGl...,Role:You are a biomedical gene nomenclature cu...
73,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,GB3S,F,NaN,21255370,True,False,...,Alias,NaN,introduction,Gb3 is the first product in the globoseries of...,NaN,"alpha 1,4-galactosyltransferase (P1PK blood gr...",Aberrant glycosylation appears to be a univers...,Shiga and the Shiga-like toxins are related pr...,PMID: 21255370\nFull Text: Aberrant glycosylat...,Role:You are a biomedical gene nomenclature cu...
33,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,F,NaN,39812643,True,False,...,Alias,NaN,introduction,BAG6 (alternatively called BAT3 or Scythe) is ...,NaN,BAG cochaperone 6,The cDNA of human BAG6 N200 was amplified by P...,The accumulation of defective polypeptides in ...,PMID: 39812643\nFull Text: The cDNA of human B...,Role:You are a biomedical gene nomenclature cu...
374,HGNC:7067,ENSG00000179583,GENE ID:4261,CIITA,NLRA,T,Prefix Gene Group Symbol,40643515,True,False,...,Alias,NaN,introduction,"Moreover, to date, five subfamilies have been ...",NLRA is the family that CIITA is in,class II major histocompatibility complex tran...,Increasing evidence reveals that the deregulat...,Increasing evidence reveals that the deregulat...,PMID: 40643515\nFull Text: Increasing evidence...,Role:You are a biomedical gene nomenclature cu...
245,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,False,False,...,NaN,_1-adrenergic receptor blocker,introduction,The AUA recommends _1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein,While the safety aspect of this meta-analysis ...,To evaluate the safety profile and efficacy of...,PMID: 18822025\nFull Text: While the safety as...,Role:You are a biomedical gene nomenclature cu...
296,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,40567688,False,False,...,NaN,global attention branch (GAB),methodology,"Therefore, a dual-branch global extraction mod...",NaN,alpha-1-B glycoprotein,"Next, the generated self-attention weight matr...",The semantic segmentation task of remote sensi...,"PMID: 40567688\nFull Text: Next, the generated...",Role:You are a biomedical gene nomenclature cu...
211,HGNC:3795,ENSG00000110203,GENE ID:2352,FOLR3,FRgamma,F,NaN,38332833,True,False,...,literature defined (uniprot protein short name),NaN,introduction,Folate receptors (FRs) are single-chain glycop...,NaN,folate receptor gamma,• Chemical stability and non-toxicity: ensurin...,Neoplasias pose a significant threat to aging ...,PMID: 38332833\nFull Text: • Chemical stabilit...,Role:You are a biomedical gene nomenclature cu...
9,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,38543574,False,False,...,NaN,primer,methods,PCR amplification of Blastocystis sp. was carr...,NaN,TUB bipartite transcription factor,Blastocystissp. is the most common intestinal ...,<i>Blastocystis</i> sp. is the most common int...,PMID: 38543574\nFull Text: Blastocystissp. is ...,Role:You are a biomedical gene nomenclature cu...
126,HGNC:2372,ENSG00000112282,GENE ID:9439,MED23,SUR-2,T,Ortholog Symbol,38987256,True,False,...,Alias,NaN,abstract,"Furthermore, we demonstrate that the nonsense ...",NaN,mediator complex subunit 23,"Reportedly,smg-1andsmg-2are involved in NMD an...",Temperature is a critical environmental cue th...,"P

In [ ]:
#Comment out after run!
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = query_using_dataframe(
    test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
)

In [160]:
#Comment out after run!
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'] = None

for idx, row in test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.iterrows():
    # Skip rows where there is no context available
    if pd.isna(row['context']):
        continue

    test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.at[
        idx, 'alt_abbrev_response'
    ] = query_claude_sonnet(row['alt_abbrev_prompt'])

    print(f'{idx} Done')

55 Done
73 Done
33 Done
374 Done
245 Done
296 Done
211 Done
9 Done
126 Done
70 Done
78 Done
180 Done
352 Done
175 Done
459 Done
289 Done
148 Done
93 Done
428 Done
77 Done


In [161]:
#Comment out after run!
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.to_excel('../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx',index=False)

In [162]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = pd.read_excel("../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx")

Testing robustness of prompt

In [198]:
# #Comment out after run!
# test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2 = query_using_dataframe(
#     test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
# )
# test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3 = query_using_dataframe(
#     test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
# )

0 Done
1 Done
2 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
10 Done
11 Done
12 Done
13 Done
14 Done
15 Done
16 Done
17 Done
18 Done
19 Done
0 Done
1 Done
2 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
10 Done
11 Done
12 Done
13 Done
14 Done
15 Done
16 Done
17 Done
18 Done
19 Done


In [204]:
# #Comment out after run!
# test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2.to_excel('../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2.xlsx',index=False)
# test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3.to_excel('../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3.xlsx',index=False)

In [205]:
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2 = pd.read_excel("../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2.xlsx")
test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3 = pd.read_excel("../output/test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3.xlsx")

### <a id='toc1_1_6_'></a>[Run all samples (available context, prioritizing full text)](#toc0_)

In [66]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df['alt_abbrev_response'] = None

# for idx, row in alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.iterrows():
#     # Skip rows where there is no context available
#     if pd.isna(row['context']):
#         continue

#     alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

0 Done
1 Done
2 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
10 Done
11 Done
12 Done
13 Done
14 Done
15 Done
16 Done
17 Done
18 Done
19 Done
20 Done
21 Done
22 Done
23 Done
24 Done
25 Done
26 Done
27 Done
28 Done
29 Done
30 Done
31 Done
32 Done
33 Done
34 Done
35 Done
36 Done
37 Done
38 Done
39 Done
40 Done
41 Done
42 Done
43 Done
44 Done
45 Done
46 Done
47 Done
48 Done
49 Done
50 Done
51 Done
52 Done
53 Done
54 Done
55 Done
56 Done
57 Done
58 Done
59 Done
60 Done
61 Done
62 Done
63 Done
64 Done
65 Done
66 Done
67 Done
68 Done
69 Done
70 Done
71 Done
72 Done
73 Done
74 Done
75 Done
76 Done
77 Done
78 Done
79 Done
80 Done
81 Done
82 Done
83 Done
84 Done
85 Done
86 Done
87 Done
88 Done
89 Done
90 Done
91 Done
92 Done
93 Done
94 Done
95 Done
96 Done
97 Done
98 Done
99 Done
100 Done
101 Done
102 Done
103 Done
104 Done
105 Done
106 Done
107 Done
108 Done
109 Done
110 Done
111 Done
112 Done
113 Done
114 Done
115 Done
116 Done
117 Done
118 Done
119 Done
120 Done
121 Done
122 Done
123

In [67]:
# #Comment out after run!
# alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.to_excel('../output/alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx',index=False)

In [8]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = pd.read_excel("../output/alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.xlsx")

In [9]:
alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured,captured as:,PMIDs from Pubtator3,Does alias represent gene in this publication?,As an Alternate Abbreviation?,...,"If not this gene, what is it?",Text Location,Relevant Text (surrounding),Notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt,alt_abbrev_response
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,7479945,yes,no,...,-,abstract,We report a mouse model of type I Usher syndro...,NaN,TUB bipartite transcription factor,NaN,Usher syndrome is a group of diseases with aut...,PMID: 7479945\nAbstract: Usher syndrome is a g...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""7479945"",\n""alias_symbol"": ""rd5"",\..."
1,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,no,...,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,These authors contributed equally to this work...,Mice homozygous for a defect of the tub (rd5) ...,PMID: 9390831\nFull Text: These authors contri...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""9390831"",\n""alias_symbol"": ""rd5"",\..."
2,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9390831,yes,no,...,-,abstract,Mice homozygous for a defect of the tub (rd5) ...,NaN,TUB bipartite transcription factor,"In this paper, we confirm that biallelic inact...",Mice homozygous for a defect of the tub (rd5) ...,"PMID: 9390831\nFull Text: In this paper, we co...",Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, here is the analys..."
3,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,9477305,yes,no,...,-,discussion,Mice homozygous for the tub (rd5) mutation exh...,NaN,TUB bipartite transcription factor,NaN,Some genetic syndromes causing loss of hearing...,PMID: 9477305\nAbstract: Some genetic syndrome...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""9477305"",\n""alias_symbol"": ""rd5"",\..."
4,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,F,NaN,36612533,no,-,...,primer,methods,The primers were as follows: forward primer RD...,NaN,TUB bipartite transcription factor,Blastocystisis one of the most common enteric ...,<i>Blastocystis</i> is one of the most common ...,PMID: 36612533\nFull Text: Blastocystisis one ...,Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, here is my analysi..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420,HGNC:9911,ENSG00000216835,GENE ID:3186,RBMXP1,HNRNP-G,F,NaN,40330012,yes,no,...,-,intro,Heterogeneous nuclear ribonucleoproteins (HNRN...,NaN,RBMX pseudogene 1,The main treatment methods of esophageal cance...,Esophageal cancer is an aggressively malignant...,PMID: 40330012\nFull Text: The main treatment ...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""40330012"",\n""alias_symbol"": ""HNRNP..."
421,HGNC:9911,ENSG00000216835,GENE ID:3186,RBMXP1,HNRNP-G,F,NaN,40330012,yes,no,...,-,intro,Heterogeneous nuclear ribonucleoproteins (HNRN...,NaN,RBMX pseudogene 1,"LW: Writing – original draft, Writing – review...",Esophageal cancer is an aggressively malignant...,PMID: 40330012\nFull Text: LW: Writing – origi...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""40330012"",\n""alias_symbol"": ""HNRNP..."
422,HGNC:9911,ENSG00000216835,GENE ID:3186,RBMXP1,HNRNP-G,F,NaN,40448182,yes,no,...,-,intro,m6A-modified readers encompass a varied range ...,NaN,RBMX pseudogene 1,NaN,"Enhancers, as distal cis-regulatory elements i...","PMID: 40448182\nAbstract: Enhancers, as distal...",Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""40448182"",\n""alias_symbol"": ""HNRNP..."
423,HGNC:9911,ENSG00000216835,GENE ID:3186,RBMXP1,HNRNP-G,F,NaN,40662138,yes,no,...,-,intro,"In addition, the structural domains of the het...",NaN,RBMX pseudogene 1,NaN,N6-methyladenosine (m6A) RNA methylation is th...,PMID: 40662138\nAbstract: N6-m

### Run all samples with prompt that doesnt use context

In [287]:
# #Comment out after run!
# consolidated_alt_abbrev_annotation_with_prompt_no_context_df['alt_abbrev_response'] = None

# for idx, row in consolidated_alt_abbrev_annotation_with_prompt_no_context_df.iterrows():

#     consolidated_alt_abbrev_annotation_with_prompt_no_context_df.at[
#         idx, 'alt_abbrev_response'
#     ] = query_claude_sonnet(row['alt_abbrev_prompt'])

#     print(f'{idx} Done')

0 Done
1 Done
2 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
10 Done
11 Done
12 Done
13 Done
14 Done
15 Done
16 Done
17 Done
18 Done
19 Done
20 Done
21 Done
22 Done


In [288]:
#Comment out after run!
consolidated_alt_abbrev_annotation_with_prompt_no_context_df.to_excel('../output/consolidated_alt_abbrev_annotation_with_prompt_no_context_df.xlsx',index=False)

In [289]:
consolidated_alt_abbrev_annotation_with_prompt_no_context_df = pd.read_excel("../output/consolidated_alt_abbrev_annotation_with_prompt_no_context_df.xlsx")

In [290]:
consolidated_alt_abbrev_annotation_with_prompt_no_context_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,gene_name,alternate_abbreviation_status,alt_abbrev_prompt,alt_abbrev_response
0,HGNC:12406,ENSG00000166402,GENE ID:7275,TUB,rd5,TUB bipartite transcription factor,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""rd5"",\n""prima..."
1,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,BAG cochaperone 6,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""SCYTHE"",\n""pr..."
2,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,"alpha-1,4-N-acetylglucosaminyltransferase",True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ALPHA4GNT"",\n..."
3,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,A4GALT1,"alpha 1,4-galactosyltransferase (P1PK blood gr...",True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""A4GALT1"",\n""p..."
4,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,GB3S,"alpha 1,4-galactosyltransferase (P1PK blood gr...",False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""GB3S"",\n""prim..."
5,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,aralkylamine N-acetyltransferase,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""SNAT"",\n""prim..."
6,HGNC:2372,ENSG00000112282,GENE ID:9439,MED23,SUR-2,mediator complex subunit 23,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""SUR-2"",\n""pri..."
7,HGNC:24086,ENSG00000148584,GENE ID:29974,A1CF,ACF,APOBEC1 complementation factor,True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ACF"",\n""prima..."
8,HGNC:24086,ENSG00000148584,GENE ID:29974,A1CF,ACF64,APOBEC1 complementation factor,True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ACF64"",\n""pri..."
9,HGNC:24086,ENSG00000148584,GENE ID:29974,A1CF,ACF65,APOBEC1 complementation factor,True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ACF65"",\n""pri..."


### LLM-as-a-judge

In [176]:
def generate_judge_prompt(alt_abbrev_prompt,alt_abbrev_response):
    """ Generate an LLM prompt for evaluating the response to the alt_abbrev_prompt for correctness.

    The generated prompt instructs the model to provide a rating score
    :param : 
    """
    prompt = f"""You will be given the original prompt and then the response.
    Your task is to provide a 'rating' scoring if the response answers the original prompt correctly.
    Give your answer on a scale of 1 to 3, where 1 means that the response is not helpful at all, and 3 means that the response is complete and correct.

    An alternate abbreviation is when an alias symbol represents the same gene name as the primary gene symbol but with different letters

    Here is the scale you should use to build your answer:
    1: Fields are missing, answer is incomplete,does not state whether it is an alternate abbreviation
    2: The evidence is irrelevant or does not support the answer
    3: The evidence currently supports the answer

    Provide your feedback as follows:
    Rating: (your rating, as a number between 1 and 3)

    You MUST provide a value for 'Rating:' in your answer.

    Now here are the question and answer.
    Prompt: {alt_abbrev_prompt}
    Response: {alt_abbrev_response}

    Provide your feedback. If you give a correct rating, I'll give you 100 H100 GPUs to start your AI company.
     """
    return prompt

In [ ]:
def generate_evaluator_prompt(alt_abbrev_prompt,alt_abbrev_response):
    """ Generate an LLM prompt for evaluating 

    The generated prompt instructs the model to provide a rating score
    :param : 
    """
    prompt = f"""Your task is to evaluate the completeness of an alias symbol representing a gene name.

    Alias gene symbol: {alias_symbol}
    Official HGNC gene name: {name}

    Split the gene name, {name}, into components by spaces, commas, and slashes.
    Count the components- "num_name_components": int()
    Map the letters in the alias, {alias_symbol}, to the components.

    Question 1: 
    Does the alias symbol have letters that do not directly represent the components of the gene name, 
    allowing each component to contribute multiple letters if the alias uses them?
    Answer Output Options:
    If yes, "extra_letters": True
    If no, "extra_letters": False , skip Question 2, and go to Question 3

    Question 2: How many letters in the alias symbol are not directly represented by the components of the gene name, 
    allowing each component to contribute multiple letters if the alias uses them?
    Answer Output Options:
    "num_extra_letters": int()
    
    Question 3: Are any of the components in the gene name not represented by the alias symbol?
    Answer Output Options:
    If yes, "missing_words": True
    If no, "missing_words": False, skip Question 4

    Question 4: How many of the components in the gene name not represented by the alias symbol?
    Answer Output Options:
    "num_missing_words": int()

    Output (Strict):
    Output only one JSON object. No markdown.
    {{
    "alias_symbol": {alias_symbol},
    "name": {name},
    "num_name_components": int(),
    "extra_letters": True or False,
    "num_extra_letters": int(),
    "missing_words": True or False,
    "num_missing_words": int()
    }}
     """
    return prompt

In [177]:
alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df = test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.copy()
alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df['judge_prompt'] = None

In [178]:
for idx, row in alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df.iterrows():
    alt_abbrev_prompt = row['alt_abbrev_prompt']
    alt_abbrev_response = row['alt_abbrev_response']
    full_prompt = generate_judge_prompt(alt_abbrev_prompt, alt_abbrev_response)

    alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df.at[idx,'judge_prompt'] = full_prompt    

In [179]:
# #Comment out after run!
# alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df['judge_prompt_response'] = None

# for idx, row in alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df.iterrows():
#     # Skip rows where there is no context available
#     if pd.isna(row['context']):
#         continue

#     alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df.at[
#         idx, 'judge_prompt_response'
#     ] = query_claude_sonnet(row['judge_prompt'])

#     print(f'{idx} Done')

0 Done
1 Done
2 Done
3 Done
4 Done
5 Done
6 Done
7 Done
8 Done
9 Done
10 Done
11 Done
12 Done
13 Done
14 Done
15 Done
16 Done
17 Done
18 Done
19 Done


In [180]:
# #Comment out after run!
# alt_abbrev_annotation_with_judge_prompt_using_fulltext_and_abstract_df.to_excel('../output/test_20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_plus_judge_df.xlsx',index=False)

In [181]:
test_20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_plus_judge_df = pd.read_excel("../output/test_20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_plus_judge_df.xlsx")

In [182]:
test_20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_plus_judge_df

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,captured_status,captured_category_list,pubtator3_pmid,alias_represent_gene_status,alternate_abbreviation_status,...,relevant_text,notes,gene_name,pmid_full_text_chunk,abstract_pmid_text,context,alt_abbrev_prompt,alt_abbrev_response,judge_prompt,judge_prompt_response
0,HGNC:17968,ENSG00000118017,GENE ID:51146,A4GNT,ALPHA4GNT,F,NaN,35959971,True,True,...,"alpha4GnT alpha1,4-N-acetylglucosaminyltransfe...",NaN,"alpha-1,4-N-acetylglucosaminyltransferase","ABBREVIATIONS:\nαGlcNAc = α1,4‐linked N ‐acety...",Gastric cancer is the second leading cause of ...,PMID: 35959971\nFull Text: ABBREVIATIONS:\nαGl...,Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, here is my analysi...",You will be given the original prompt and then...,Rating: 3\n\nThe response is excellent and cor...
1,HGNC:18149,ENSG00000128274,GENE ID:53947,A4GALT,GB3S,F,NaN,21255370,True,False,...,Gb3 is the first product in the globoseries of...,NaN,"alpha 1,4-galactosyltransferase (P1PK blood gr...",Aberrant glycosylation appears to be a univers...,Shiga and the Shiga-like toxins are related pr...,PMID: 21255370\nFull Text: Aberrant glycosylat...,Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, I do not see evide...",You will be given the original prompt and then...,Rating: 3\n\nThe response is excellent and cor...
2,HGNC:13919,"ENSG00000234651, ENSG00000096155, ENSG00000233...",GENE ID:7917,BAG6,SCYTHE,F,NaN,39812643,True,False,...,BAG6 (alternatively called BAT3 or Scythe) is ...,NaN,BAG cochaperone 6,The cDNA of human BAG6 N200 was amplified by P...,The accumulation of defective polypeptides in ...,PMID: 39812643\nFull Text: The cDNA of human B...,Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, I did not find any...",You will be given the original prompt and then...,Rating: 3\n\nThe response is excellent. It cor...
3,HGNC:7067,ENSG00000179583,GENE ID:4261,CIITA,NLRA,T,Prefix Gene Group Symbol,40643515,True,False,...,"Moreover, to date, five subfamilies have been ...",NLRA is the family that CIITA is in,class II major histocompatibility complex tran...,Increasing evidence reveals that the deregulat...,Increasing evidence reveals that the deregulat...,PMID: 40643515\nFull Text: Increasing evidence...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""40643515"",\n""alias_symbol"": ""NLRA""...",You will be given the original prompt and then...,Rating: 3\n\nThe response is excellent and cor...
4,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,A1B,F,NaN,18822025,False,False,...,The AUA recommends _1-adrenergic receptor bloc...,NaN,alpha-1-B glycoprotein,While the safety aspect of this meta-analysis ...,To evaluate the safety profile and efficacy of...,PMID: 18822025\nFull Text: While the safety as...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""18822025"",\n""alias_symbol"": ""A1B"",...",You will be given the original prompt and then...,Rating: 3\n\nThe response correctly identifies...
5,HGNC:5,ENSG00000121410,GENE ID:1,A1BG,GAB,F,NaN,40567688,False,False,...,"Therefore, a dual-branch global extraction mod...",NaN,alpha-1-B glycoprotein,"Next, the generated self-attention weight matr...",The semantic segmentation task of remote sensi...,"PMID: 40567688\nFull Text: Next, the generated...",Role:You are a biomedical gene nomenclature cu...,"Based on the provided text, I do not see any e...",You will be given the original prompt and then...,Rating: 3\n\nThe response correctly identifies...
6,HGNC:3795,ENSG00000110203,GENE ID:2352,FOLR3,FRgamma,F,NaN,38332833,True,False,...,Folate receptors (FRs) are single-chain glycop...,NaN,folate receptor gamma,• Chemical stability and non-toxicity: ensurin...,Neoplasias pose a significant threat to aging ...,PMID: 38332833\nFull Text: • Chemical stabilit...,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": ""38332833"",\n""alias_symbol"": ""FRgam...",You 

### <a id='toc1_1_7_'></a>[Analysis](#toc0_)

In [142]:
def extract_json_block(s):
    if pd.isna(s) or not str(s).strip():
        return np.nan

    s = str(s)

    stack = []
    start = None

    for i, ch in enumerate(s):
        if ch in '{[':
            stack.append(ch)
            if start is None:
                start = i
        elif ch in '}]' and stack:
            stack.pop()
            if not stack:
                return s[start:i+1]

    return np.nan


def parse_response(x):
    if pd.isna(x) or not str(x).strip():
        return np.nan

    s = str(x)

    block = extract_json_block(s)
    if not block:
        return np.nan

    try:
        return json.loads(block)
    except json.JSONDecodeError:
        return np.nan

In [143]:
def extract_representation_status(d):
    if not isinstance(d, dict):
        # response was empty / invalid
        return np.nan

    if "alias_symbol_represents_primary_gene" not in d:
        return np.nan

    if d["alias_symbol_represents_primary_gene"] == "":
        # explicit empty string in JSON
        return "-"

    return d["alias_symbol_represents_primary_gene"]

In [144]:
def extract_relationship_type(d):
    if not isinstance(d, dict):
        # response was empty / invalid
        return np.nan

    if "relationship_type" not in d:
        return np.nan

    if d["relationship_type"] == "":
        # explicit empty string in JSON
        return "-"

    return d["relationship_type"]

In [145]:
def extract_alternate_abbreviation(d):
    if not isinstance(d, dict):
        return np.nan

    val = d.get("alternate_abbreviation", None)

    if val is None or val == "":
        return np.nan

    return str(val).strip().upper() == "TRUE"

In [146]:
def create_analysis_summary(column: pd.Series):
    """ Summarize value counts with:
      - True/False percentages over non-NaN denominator
      - NaN percentage over total rows

    :param column: Dataframe and column to analyze (Example: df["column"])
    :return: dataframe with summarized results
    """
    counts = (
        column.value_counts(dropna=False)
              .rename("numerator")
              .rename_axis("consensus_w_curator")
    )

    non_nan_denominator = column.notna().sum()
    total_denominator = len(column)

    return (
        counts
        .reset_index()
        .assign(
            denominator=lambda df: df["consensus_w_curator"].apply(
                lambda x: total_denominator if pd.isna(x) else non_nan_denominator
            ),
            percentage=lambda df: df["numerator"] / df["denominator"] * 100
        )
    )

In [147]:
def consolidate_match(s):
    if s.eq(True).any():
        return True
    if s.eq(False).any():
        return False
    return np.nan

In [148]:
def consolidate_bool(series):
    s = series.dropna()
    if s.empty:
        return np.nan
    if s.any():
        return True
    return False

In [149]:
def consolidate_group(group):
    s = group["response_alternate_abbreviation"].dropna()

    if s.empty:
        return pd.Series({
            "As an Alternate Abbreviation?": group["As an Alternate Abbreviation?"].iloc[0],
            "response_alternate_abbreviation": np.nan,
            "response_evidence": np.nan,
        })

    if s.any():
        # pick first True row
        row = group.loc[group["response_alternate_abbreviation"] == True].iloc[0]
        return pd.Series({
            "As an Alternate Abbreviation?": group["As an Alternate Abbreviation?"].iloc[0],
            "response_alternate_abbreviation": True,
            "response_evidence": row["response_evidence"],
        })

    # otherwise all False
    row = group.loc[group["response_alternate_abbreviation"] == False].iloc[0]
    return pd.Series({
        "As an Alternate Abbreviation?": group["As an Alternate Abbreviation?"].iloc[0],
        "response_alternate_abbreviation": False,
        "response_evidence": row["response_evidence"],
    })

In [150]:
def boolean_confusion_matrix(df, gt_col, pred_col):
    """
    Compute a boolean confusion matrix in the traditional matrix style.

    Parameters:
    - df: pandas DataFrame
    - gt_col: str, name of the ground truth column
    - pred_col: str, name of the predicted column

    Returns:
    - confusion_matrix: pandas DataFrame with row/column labels
    """
    gt = df[gt_col]
    pred = df[pred_col]

    # Keep only valid predictions
    mask = pred.notna()
    gt = gt[mask]
    pred = pred[mask]

    # Build crosstab
    cm = pd.crosstab(gt, pred, dropna=False)

    # Reindex to ensure consistent order
    cm = cm.reindex(index=[False, True], columns=[False, True], fill_value=0)

    # Set row and column labels for display
    cm.index.name = "Ground Truth"
    cm.columns.name = "Predicted"

    return cm

In [228]:
def analyze_alt_abbrev_results(    
    df,
    extract_evidence=True,
    consolidate=True,
    groupby_col="pubtator3_pmid",
):
    analysis_df = df.copy()

    # Extract and parse
    analysis_df["alt_abbrev_response_extracted"] = (
        analysis_df["alt_abbrev_response"].apply(extract_json_block)
    )

    analysis_df["alt_abbrev_response_parsed"] = (
        analysis_df["alt_abbrev_response_extracted"].apply(parse_response)
    )
    analysis_df["response_alternate_abbreviation"] = (
        analysis_df["alt_abbrev_response_parsed"]
        .apply(extract_alternate_abbreviation)
    )
    ## optional: extract evidence
    if extract_evidence:
        analysis_df["response_evidence"] = (
            analysis_df["alt_abbrev_response_parsed"]
            .apply(lambda d: d.get("evidence") if isinstance(d, dict) else None)
        )

    # Optional: Consolidate per PMID
    if consolidate:
        consolidated_df = (
            analysis_df
            .groupby(groupby_col, as_index=False)
            .agg({
                "alternate_abbreviation_status": "first",
                "response_alternate_abbreviation": consolidate_bool,
            })
        )
    else:
        consolidated_df = analysis_df


    # Compare to manual curation
    consolidated_df["relationship_type_matches"] = (
        consolidated_df["alternate_abbreviation_status"]
        == consolidated_df["response_alternate_abbreviation"]
    )

    # Summarize
    relationship_type_match_analysis_summary = create_analysis_summary(
        consolidated_df["relationship_type_matches"]
    )

    # Create confusion matrix
    cm = boolean_confusion_matrix(
        consolidated_df,
        "alternate_abbreviation_status",
        "response_alternate_abbreviation",
    )

    TN = cm.loc[False, False]
    FP = cm.loc[False, True]
    FN = cm.loc[True, False]
    TP = cm.loc[True, True]

    # Compute metrics
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    metrics_df = pd.DataFrame({
        "precision": [precision],
        "recall": [recall],
        "f1": [f1],
        "TP": [TP],
        "FP": [FP],
        "TN": [TN],
        "FN": [FN],
    })


    return analysis_df, consolidated_df, relationship_type_match_analysis_summary, cm, metrics_df


#### <a id='toc1_1_7_3_'></a>[Analysis (20 samples, available context, prioritizing full text)](#toc0_)

In [194]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df, consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df, relationship_type_match_analysis_summary, cm, metrics_df = analyze_alt_abbrev_results(
    test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
)

In [188]:
relationship_type_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,True,17,20,85.0
1,False,3,20,15.0


In [189]:
cm

Predicted,False,True
Ground Truth,,
False,16,2
True,1,1


In [195]:
metrics_df

,precision,recall,f1,TP,FP,TN,FN
0,0.333333,0.5,0.4,1,2,16,1


Analyzing robustness of prompt

In [206]:
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2, consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2, relationship_type_match_analysis_summary2, cm2, metrics_df2 = analyze_alt_abbrev_results(
    test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2
)
analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3, consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3, relationship_type_match_analysis_summary3, cm3, metrics_df3 = analyze_alt_abbrev_results(
    test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3
)

In [207]:
df1_aligned = consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.set_index("pubtator3_pmid")
df2_aligned = consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df2.set_index("pubtator3_pmid")
df3_aligned = consolidated_analysis_test20_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df3.set_index("pubtator3_pmid")

df2_aligned = df2_aligned.reindex(df1_aligned.index)
df3_aligned = df3_aligned.reindex(df1_aligned.index)

agreement_rate = (
    (df1_aligned["response_alternate_abbreviation"] == df2_aligned["response_alternate_abbreviation"]) &
    (df1_aligned["response_alternate_abbreviation"] == df3_aligned["response_alternate_abbreviation"])
).mean()

In [208]:
agreement_rate

1.0

#### <a id='toc1_1_7_4_'></a>[Analyze (available context, prioritizing full text)](#toc0_)

In [33]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.copy()
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response_extracted"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response"].apply(extract_json_block)
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response_parsed"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response_extracted"].apply(parse_response)

In [34]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["response_alternate_abbreviation"] = analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response_parsed"].apply(
    extract_alternate_abbreviation
)

In [35]:
analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["response_evidence"] = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["alt_abbrev_response_parsed"]
    .apply(lambda d: d.get("evidence") if isinstance(d, dict) else None)
)

In [38]:
consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df = (
    analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df
    .groupby("PMIDs from Pubtator3", as_index=False)
    .apply(consolidate_group))

/var/folders/vt/znzp_c6s02q6kjcmqfk0cb500000gq/T/ipykernel_6682/3822719725.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(consolidate_group))


In [40]:
col_a = consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["As an Alternate Abbreviation?"]
col_b = consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["response_alternate_abbreviation"]



consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["relationship_type_matches"] = (
    (col_a == col_b)
)

In [41]:
consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df

,PMIDs from Pubtator3,As an Alternate Abbreviation?,response_alternate_abbreviation,response_evidence,relationship_type_matches
0,2447112,yes,False,Alpha 1-beta glycoprotein (A1B) was purified t...,False
1,6322416,-,False,,False
2,7277944,-,False,,False
3,7479945,no,False,,False
4,7486253,-,False,,False
...,...,...,...,...,...
168,40721578,no,False,,False
169,40819127,no,False,,False
170,40890122,no,False,,False
171,40894640,-,False,,False


In [42]:
relationship_type_match_analysis_summary = create_analysis_summary(
    consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df["relationship_type_matches"]
)
relationship_type_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,False,173,173,100.0


In [232]:
df = consolidated_analysis_analysis_alt_abbrev_annotation_with_prompt_using_fulltext_and_abstract_df.copy()

gt = df["alternate_abbreviation_status"]
pred = df["response_alternate_abbreviation"]


In [233]:
valid = pred.notna()
gt = gt[valid]
pred = pred[valid]

In [234]:
correct = gt == pred
correct.value_counts()

True     133
False     40
Name: count, dtype: int64

In [235]:
confusion_matrix = pd.crosstab(
    gt,
    pred,
    rownames=["Ground Truth"],
    colnames=["Predicted"]
).reindex(
    index=["NO", "YES"],
    columns=["NO", "YES"],
    fill_value=0
)

confusion_matrix

Predicted,NO,YES
Ground Truth,,
NO,119,31
YES,8,14


In [236]:
TN = confusion_matrix.loc["NO", "NO"]
FP = confusion_matrix.loc["NO", "YES"]
FN = confusion_matrix.loc["YES", "NO"]
TP = confusion_matrix.loc["YES", "YES"]

In [237]:
#Of all predicted positives, how many were actually correct?
precision = TP / (TP + FP) if (TP + FP) else np.nan
#Of all actual positives, how many were correctly predicted?
recall = TP / (TP + FN) if (TP + FN) else np.nan
#The balance between precision and recall
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else np.nan

In [238]:
precision

0.3111111111111111

0.31 means there are a lot of false positives, model overestimates what should be an alternate abbreviation. need to make definition more restrictive

In [239]:
recall

0.6363636363636364

0.64 means that a little more than half of the ground truth positives were predicted correctly. need to look at the ones that were not predicted correctly so that when I make definition more restrictive, I clarify what is knocking out these samples

In [240]:
f1

0.417910447761194

#### Analyze the results from using prompt with no context

In [291]:
analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df, _, relationship_type_match_analysis_summary, cm, metrics_df = analyze_alt_abbrev_results(
    consolidated_alt_abbrev_annotation_with_prompt_no_context_df, extract_evidence=False, consolidate=False
)

In [292]:
relationship_type_match_analysis_summary

,consensus_w_curator,numerator,denominator,percentage
0,True,19,23,82.608696
1,False,4,23,17.391304


In [293]:
cm

Predicted,False,True
Ground Truth,,
False,12,3
True,1,7


In [295]:
metrics_df

,precision,recall,f1,TP,FP,TN,FN
0,0.7,0.875,0.777778,7,3,12,1


In [297]:
false_positives = analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df.loc[(analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df["alternate_abbreviation_status"] == False) & (analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df["response_alternate_abbreviation"] == True)]
false_positives

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,gene_name,alternate_abbreviation_status,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_extracted,alt_abbrev_response_parsed,response_alternate_abbreviation,relationship_type_matches
5,HGNC:19,ENSG00000129673,GENE ID:15,AANAT,SNAT,aralkylamine N-acetyltransferase,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""SNAT"",\n""prim...","{\n""pmid"": """",\n""alias_symbol"": ""SNAT"",\n""prim...","{'pmid': '', 'alias_symbol': 'SNAT', 'primary_...",True,False
12,HGNC:32629,ENSG00000222370,GENE ID:677818,SNORA36B,ACA36b,"small nucleolar RNA, H/ACA box 36B",False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ACA36b"",\n""pr...","{\n""pmid"": """",\n""alias_symbol"": ""ACA36b"",\n""pr...","{'pmid': '', 'alias_symbol': 'ACA36b', 'primar...",True,False
13,HGNC:3795,ENSG00000110203,GENE ID:2352,FOLR3,FRgamma,folate receptor gamma,False,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""FRgamma"",\n""p...","{\n""pmid"": """",\n""alias_symbol"": ""FRgamma"",\n""p...","{'pmid': '', 'alias_symbol': 'FRgamma', 'prima...",True,False


In [298]:
false_positives = analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df.loc[(analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df["alternate_abbreviation_status"] == True) & (analysis_consolidated_alt_abbrev_annotation_with_prompt_no_context_df["response_alternate_abbreviation"] == False)]
false_positives

,HGNC_ID,ENSG_ID,NCBI_ID,primary_gene_symbol,gene_symbol,gene_name,alternate_abbreviation_status,alt_abbrev_prompt,alt_abbrev_response,alt_abbrev_response_extracted,alt_abbrev_response_parsed,response_alternate_abbreviation,relationship_type_matches
10,HGNC:24086,ENSG00000148584,GENE ID:29974,A1CF,ASP,APOBEC1 complementation factor,True,Role:You are a biomedical gene nomenclature cu...,"{\n""pmid"": """",\n""alias_symbol"": ""ASP"",\n""prima...","{\n""pmid"": """",\n""alias_symbol"": ""ASP"",\n""prima...","{'pmid': '', 'alias_symbol': 'ASP', 'primary_g...",False,False
